# Sentiment Analysis 

This notebook implements sentiment analysis, to assign a numeric value to the sentiment of comments. 

In [3]:
import numpy as np
import regex

from transformers import BertTokenizerFast, pipeline

from database.comments import Comments

import sys
sys.path.append('../pipeline')
from nlp_tasks import NLP_Tasks

Can use the sentiment analysis pipeline from HuggingFace

In [4]:
cs = Comments(env='dev')

Connecting to the ai4ci-db-dev database...
Successfully connected to ai4ci-db-dev.


In [ ]:
df = cs.read_all()
df = df[:1000]
print('df shape:', df.shape)

df shape: (1000, 14)


In [ ]:
df.head()

,id,council,comment_id,application_id,address,stance,date,comment_text,add_date,lat,lon,cleaned_comment_text,lsoa_code,sentiment_score
0,74673,Westminster,21/06791/FULL_6,21/06791/FULL,None,Objects,2022-02-26,Dear Sirs\n\nWe object to the application made...,2025-04-07,NaN,NaN,Dear Sirs\n\nWe object to the application made...,None,None
1,74675,Westminster,21/06791/FULL_8,21/06791/FULL,None,Objects,2021-11-10,"Hi,\nI strongly object to these works. I've be...",2025-04-07,NaN,NaN,"Hi,\nI strongly object to these works. I've be...",None,None
2,74676,Westminster,21/06791/FULL_9,21/06791/FULL,None,Objects,2021-11-10,"I would like to object to this proposal, in pa...",2025-04-07,NaN,NaN,"I would like to object to this proposal, in pa...",None,None
3,74678,Westminster,21/06791/FULL_11,21/06791/FULL,None,Objects,2021-10-31,I live in the Garden Flat that backs onto the ...,2025-04-07,NaN,NaN,I live in the Garden Flat that backs onto the ...,None,None
4,71169,Barnet,24/0152/FUL_3,24/0152/FUL,"Flat 7, Graham House Barnet London EN4 5EX",Objects,2024-01-24,"Dear Barnet Council,\nI know the site well as ...",2025-04-04,51.500021,-0.192442,"Dear Council,\nI know the site well as my moth...",E09000020,None


In [ ]:
def split_period_or_length(comment, max_length=100):
    sentences = regex.split(r'(?<=[.!?]) +', comment)
    split_sentences = []
    for sentence in sentences:
        if len(sentence) > max_length:
            # Split long sentences into smaller chunks
            for i in range(0, len(sentence), max_length):
                split_sentences.append(sentence[i:i + max_length])
        else:
            split_sentences.append(sentence)
    return split_sentences

In [ ]:
sentiment_model = pipeline(model="finiteautomata/bertweet-base-sentiment-analysis")

def sentiment_score(comment, stance):
    
    # Split the comment into sentences
    # sentences = regex.split(r'(?<=[.!?]) +', comment)
    sentences = split_period_or_length(comment, max_length=100)
    n = len(sentences)

    # Analyse sentiment for each sentence
    sentiment_results = sentiment_model(sentences)

    # Calculate score by adding 'POS' scores and subtracting 'NEG' scores
    score = 0
    for result in sentiment_results:
        if result['label'] == 'POS':
            score += result['score']
        elif result['label'] == 'NEG':
            score -= result['score']

    score = float(score / n)

    # Adjust score based on stance
    if stance == 'Objects' and score > 0:
        score = 0
    elif stance == 'Supports' and score < 0:
        score = 0

    return score

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0
Device set to use mps:0


In [ ]:
i=2
id = df['comment_id'][i]
score = sentiment_score(df['cleaned_comment_text'][i], df['stance'][i])

In [ ]:
cs.update_sentiment_score_by_comment_id(id, score)

Successfully updated sentiment_score for comment_id 21/06791/FULL_9: -0.3643794357776642


True

In [ ]:
nlp = NLP_Tasks()